# 032 · Introduction to Optimizers — Why Plain Gradient Descent Is Not Enough

Every optimizer in this module is a modification of one line:

$$w \leftarrow w - \eta \, \partial L / \partial w$$

**None of them replaces backpropagation.** They change what you do with the
gradient once you have it. This notebook demonstrates the four problems they
were each invented to attack.

| Part | The problem |
|---|---|
| A | choosing η is hard, and the right value **changes during training** |
| B | one η for **every** weight, when different directions want different steps |
| C | local minima |
| D | **saddle points** — and why they dominate in high dimensions |

Needs `numpy`.

In [ ]:
import numpy as np

## Part A — The learning rate is hard to choose

Take the simplest possible problem, `L = 0.5 w²`, and sweep η.

In [ ]:
def descend(lr, steps=50, w0=10.0):
    w, path = w0, []
    for _ in range(steps):
        path.append(w)
        w = w - lr * w              # dL/dw = w
    return np.array(path)


print(f"{'lr':>7}{'final |w|':>14}{'verdict':>16}")
for lr in (0.001, 0.01, 0.1, 0.5, 1.0, 1.9, 2.0, 2.1):
    p = descend(lr)
    final = abs(p[-1])
    verdict = ("diverged" if not np.isfinite(final) or final > 10
               else "crawling" if final > 1
               else "good")
    print(f"{lr:>7}{final:>14.4f}{verdict:>16}")

print("\nThe usable band is narrow, and it is bounded: above lr = 2/curvature")
print("this problem diverges outright. You cannot find that band without trying.")

In [ ]:
# Worse: the RIGHT rate changes with the STAGE of training.
p_fast = descend(0.5, steps=50)
print("With lr = 0.5, |w| by step:")
for s in (0, 1, 5, 10, 20, 40, 49):
    print(f"  step {s:>2}: {abs(p_fast[s]):.2e}")

print("\nEarly on you want big steps to cover ground. Later you want small ones")
print("to settle. A single fixed number cannot be both.")
print("\nScheduling helps - but a schedule is decided BEFORE training, so it")
print("cannot react to what actually happens.")

## Part B — One learning rate for every weight

Different directions in the loss surface have wildly different curvature. A rate
that is safe for the steep one is far too small for the shallow one.

In [ ]:
# L = 0.5(x^2 + 20 y^2): the y direction is 20x steeper.
A, B = 1.0, 20.0

print(f"steep axis (y) diverges above lr = {2/B}")
print(f"shallow axis (x) would happily take lr = {2/A}")
print(f"-> the shallow axis is forced to run {(2/A)/(2/B):.0f}x slower than it could\n")

def run(lr, steps=60):
    p = np.array([9.0, 1.0])
    for _ in range(steps):
        p = p - lr * np.array([A * p[0], B * p[1]])
    return p

for lr in (0.02, 0.09, 0.11):
    p = run(lr)
    ok = "diverged" if not np.all(np.isfinite(p)) or np.abs(p).max() > 1e3 else "stable"
    print(f"  lr = {lr:<5} -> x = {p[0]:>10.4f}   y = {p[1]:>10.2e}   {ok}")

print("\nThe steep axis caps the rate. That same cap starves the shallow one,")
print("which is still at x = 2.7 after 60 steps. AdaGrad and RMSprop attack this.")

## Part C — Local minima

A real dip in the surface, but not the deepest one. Gradient descent has no way
to know the difference, because it only ever looks at the slope where it stands.

In [ ]:
# A surface with one shallow local minimum and one deeper global minimum.
def L(w):
    return w**4 - 3*w**3 + 2      # dips near w = 0 and w = 2.25

def dL(w):
    return 4*w**3 - 9*w**2

for start in (-1.0, 0.5, 1.0, 3.0):
    w = start
    for _ in range(500):
        w = w - 0.01 * dL(w)
    print(f"  start {start:>5} -> settled at w = {w:>7.4f}, loss = {L(w):>8.4f}")

print("\nWhere you start decides where you end. Nothing in the update rule")
print("can see past the local slope.")

## Part D — Saddle points, and why they matter more

A saddle goes **up** in one direction and **down** in another, with a flat
plateau in between where the gradient is ≈ 0. From the inside it is
indistinguishable from a minimum: **gradient ≈ 0, so the update ≈ 0, and training
looks converged when it has stalled.**

In [ ]:
# The simplest saddle: L = x^2 - y^2, critical point at the origin.
def grad_saddle(p):
    return np.array([2 * p[0], -2 * p[1]])

p = np.array([1.0, 1e-6])        # start almost exactly on the ridge
for step in range(200):
    p = p - 0.05 * grad_saddle(p)
    if step in (0, 20, 60, 199):
        g = grad_saddle(p)
        print(f"  step {step:>3}: p = ({p[0]:>9.2e}, {p[1]:>9.2e})  "
              f"|grad| = {np.linalg.norm(g):.2e}")

print("\nThe gradient is tiny for a long stretch and the point barely moves.")
print("Escape depends entirely on the small y component growing.")

### Why saddles dominate in high dimensions

A critical point is a **minimum** only if the surface curves *up* in **every**
direction — all Hessian eigenvalues positive. With `n` parameters that is `n`
conditions that must all hold at once. A saddle needs only one to fail.

Check the scaling empirically on random symmetric matrices.

In [ ]:
rng = np.random.default_rng(0)

def classify(n, trials=400):
    """Fraction of random critical points that are minima, maxima or saddles."""
    mins = maxs = 0
    for _ in range(trials):
        M = rng.standard_normal((n, n))
        H = (M + M.T) / 2                     # a random symmetric Hessian
        ev = np.linalg.eigvalsh(H)
        if (ev > 0).all():
            mins += 1
        elif (ev < 0).all():
            maxs += 1
    return mins / trials, maxs / trials, 1 - (mins + maxs) / trials


print(f"{'dimensions':>11}{'minima':>10}{'maxima':>10}{'saddles':>10}")
for n in (1, 2, 3, 5, 10, 20):
    lo, hi, sad = classify(n)
    print(f"{n:>11}{lo:>10.3f}{hi:>10.3f}{sad:>10.3f}")

print("\nBy 10 dimensions essentially every critical point is a saddle. A real")
print("network has millions of parameters. This is why the field stopped")
print("worrying about local minima and started worrying about saddle points.")

In [ ]:
lo, _, sad = classify(20)
assert sad > 0.95        # saddles dominate by 20 dimensions
print(f"at 20 dimensions: {sad:.1%} of critical points are saddles")
print("\nNOTE: these are RANDOM symmetric matrices, not real loss Hessians.")
print("The argument is about counting conditions, and the trend is the point -")
print("this is not a measurement of any actual network's loss surface.")

## What to take away

- **Every optimizer is a modification of `w ← w − η ∂L/∂w`** — none replaces
  backpropagation.
- **Problem 1:** η is hard to choose, and the right value changes with dataset,
  architecture and **stage of training**. Scheduling helps but is fixed in
  advance.
- **Problem 2: one η for every weight.** The steep direction caps the rate and
  starves the shallow one.
- **Problem 3: local minima** — a real dip, but not the deepest.
- **Problem 4: saddle points** — up one way, down another, flat in between.
- **Saddle points are far more common than local minima in high dimensions**, and
  the counting argument above shows why: a minimum needs *every* direction to
  curve up.
- Both look identical from inside: **gradient ≈ 0, update ≈ 0**, and training
  looks converged when it has stalled.
- **Momentum and NAG attack getting stuck; AdaGrad and RMSprop attack the single
  rate; Adam combines both.**

## Exercises

1. In Part A, find the exact learning rate at which `L = 0.5w²` starts to
   diverge, and derive it from the curvature. Then predict the threshold for
   `L = 0.5·c·w²` and check it.
2. Part C's outcome depended on the starting point. Map the basin boundary: for
   which starting values of `w` does gradient descent find the global minimum?
3. Add a little noise to each update in Part C (this is what the "S" in SGD gives
   you). Does it escape the shallow minimum? How much noise is needed?
4. Part D used random symmetric matrices. Build an actual tiny network, compute
   its Hessian at a converged point, and look at the eigenvalue signs. How many
   are near zero, and what does that suggest about the surface?
5. Skip ahead: implement momentum and rerun Part D's saddle. Does the velocity
   term help it escape, and if so why?